### Creating vector data with respect to the description of the books

In [1]:
import os

In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_chroma import Chroma

In [4]:
# Using the HuggingFace word Embedding model to create the vector database for the books

from langchain_huggingface import HuggingFaceEmbeddings

model_name = "sentence-transformers/all-MiniLM-L6-v2"
model_kwargs = {"device": "cpu"}
encode_kwargs = {"normalize_embeddings": False}

hf_embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [5]:
import pandas as pd

books = pd.read_csv("../Dataset/cleaned_books.csv")

In [6]:
# New column containing the combined value of the ISBN number and the description of the book

books["tagged_description"]

0       9780002005883 A NOVEL THAT READERS and critics...
1       9780002261982 A new 'Christie for Christmas' -...
2       9780006178736 A memorable, mesmerizing heroine...
3       9780006280897 Lewis' work on the nature of lov...
4       9780006280934 "In The Problem of Pain, C.S. Le...
                              ...                        
5192    9788172235222 On A Train Journey Home To North...
5193    9788173031014 This book tells the tale of a ma...
5194    9788179921623 Wisdom to Create a Life of Passi...
5195    9788185300535 This collection of the timeless ...
5196    9789027712059 Since the three volume edition o...
Name: tagged_description, Length: 5197, dtype: str

In [7]:
# Storing the new column into a text file so that we can use it later to create the vector data for the books

books["tagged_description"].to_csv(
    "../Saved Files/tagged_description.txt",
    sep="\n",
    index=False,
    header=False
)

In [8]:
# Loading the text file using the TextLoader class from the langchain_community.document_loaders module

raw_documents = TextLoader("../Saved Files/tagged_description.txt").load()

In [9]:
# Initializing the CharacterTextSplitter with the following parameters:

text_splitter = CharacterTextSplitter(
    chunk_size=1,           # Chunk size means the max number of characters in each chunk
    chunk_overlap=0,        # Chunk overlap means the number of characters that will be repeated in each chunk
    separator="\n"          # Separator means the character that will be used to split the text into chunks
)

In [10]:
import pickle

pickle_file = "../Saved Files/split_documents.pkl"

if os.path.exists(pickle_file):
    print("Loading cached split documents...")
    with open(pickle_file, "rb") as f:
        documents = pickle.load(f)
else:
    print("Splitting raw documents and saving...")
    documents = text_splitter.split_documents(raw_documents)

    with open(pickle_file, "wb") as f:
        pickle.dump(documents, f)

Loading cached split documents...


In [11]:
# Checking the first document in the list of documents

documents[0]

Document(metadata={'source': 'tagged_description.txt'}, page_content='9780002005883 A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a preacher and the grandson (both maternal and paternal) of preachers. It’s 1956 in Gilead, Iowa, towards the end of the Reverend Ames’s life, and he is absorbed in recording his family’s story, a legacy for the young son he will never see grow up. Haunted by his grandfather’s presence, John tells of the rift between his grandfather and his father: the elder, an angry visionary who fought for the abolitionist cause, and his son, an ardent pacifist. He is troubled, too, by his prodigal namesake, Jack (John Ames) Boughton, his best friend’s lost son who returns to Gilead searching for forgiveness and redemption. Told in John Ames’s joyous, rambling voice that finds beauty, humour and truth in the smallest of life’s details, Gi

In [12]:
persist_directory = "../Saved Files/chroma_db"

# Creating a vector embedding and storing it using Chroma
if os.path.exists(persist_directory):
    print("Loading existing vector database...")
    db_books = Chroma(
        persist_directory=persist_directory,
        embedding_function=hf_embeddings
    )
else:
    print("Creating and saving new vector database...")
    db_books = Chroma.from_documents(
        documents=documents,
        embedding=hf_embeddings,
        persist_directory=persist_directory
    )

Loading existing vector database...


In [13]:
# Creating a query to pass to the vector database to get the most similar books to the query

query = "A book to teach children about nature"

In [14]:
# Passing the query to the vector db and searching for the most similar books to the query.
# The k parameter is set to 10, which means that the top 10 most similar books will be returned.

docs = db_books.similarity_search(
    query=query,
    k = 10
)
docs

[Document(id='79d04130-3e4c-4e9e-ac19-d8fd2f2fa0a0', metadata={'source': 'tagged_description.txt'}, page_content='9780786808069 Children will discover the exciting world of their own backyard in this introduction to familiar animals from cats and dogs to bugs and frogs. The combination of photographs, illustrations, and fun facts make this an accessible and delightful learning experience.'),
 Document(id='05b13f2e-01bb-4400-ba8a-62b354f48429', metadata={'source': 'tagged_description.txt'}, page_content="9780786808380 Introduce your babies to birds, cats, dogs, and babies through fine art, illustration, and photographs. These books are a rare opportunity to expose little ones to a range of images on a single subject, from simple child's drawings and abstract art to playful photos. A brief text accompanies each image, introducing the baby to some basic -- and sometimes playful -- information about the subjects."),
 Document(id='a232a571-fe68-4acc-a16f-a0fed66da147', metadata={'source': '